# Mô-đun 3: Offer Assessment System

**Đồ án:** Hệ hỗ trợ quyết định trong định giá lương và tư vấn đàm phán lương  
**Module:** Offer Assessment System  
**Mục tiêu:** Đánh giá chất lượng offer bằng kiến trúc 3 tầng: Rule Engine → Ordinal Classification → Cost-sensitive Decision Layer

---

## Tổng quan kiến trúc 3 tầng

```
Đầu vào: Hồ sơ ứng viên + current_offer + Kết quả Module 1 + Kết quả Module 2
        ↓
┌─────────────────────────────────────────────────────────┐
│  TẦNG 1 — Rule-Based Decision Engine                    │
│  • Tính gap so với pred_Q10/Q50/Q75/Q90                 │
│  • Tính gap so với peer_Q10/median/Q75/Q90              │
│  • Gán nhãn: UNDERPAID / LOW / FAIR / GOOD / EXCELLENT  │
│  • Tính rule_score ∈ [0, 4]                             │
├─────────────────────────────────────────────────────────┤
│  TẦNG 2 — Ordinal Classification Model                  │
│  • Feature Engineering đầy đủ (offer gaps + ratios)    │
│  • Ordinal Logistic Regression (mitas_logistic) hoặc   │
│    Random Forest Classifier (fallback)                  │
│  • Output: P(UNDERPAID), P(LOW), P(FAIR), P(GOOD),     │
│            P(EXCELLENT) cho từng hồ sơ                  │
├─────────────────────────────────────────────────────────┤
│  TẦNG 3 — Cost-Sensitive Decision Layer                 │
│  • Cost Matrix (chi phí sai lầm không đối xứng)        │
│  • Expected Cost & Expected Utility                     │
│  • Quyết định cuối: argmin(Expected Cost)               │
│  • Confidence Score từ xác suất phân phối              │
└─────────────────────────────────────────────────────────┘
        ↓
Đầu ra: offer_quality_label, offer_quality_probability,
        decision, confidence_score, expected_cost, expected_utility
```

## Các bước chính

1. Load dữ liệu + kết quả từ Module 1 & 2  
2. Feature Engineering (gap features, ratio features)  
3. Tầng 1: Rule Engine → gán nhãn + rule_score  
4. Tầng 2: Ordinal Classification (Ordinal LR + RF benchmark)  
5. Tầng 3: Cost-Sensitive Decision Layer  
6. Đánh giá mô hình (Accuracy, Macro F1, ROC-AUC, Confusion Matrix)  
7. SHAP Explainability  
8. Visualization đầy đủ  
9. Lưu model và kết quả  


In [ ]:
import sys
print(sys.executable)
print(sys.version)


## 1. Import thư viện

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
import os
import json
from typing import Dict, List, Tuple, Optional

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc
)

# ── LightGBM (benchmark) ──────────────────────────────────────────────────────
try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("LightGBM không khả dụng — bỏ qua benchmark LGBM Classifier.")

# ── SHAP ──────────────────────────────────────────────────────────────────────
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("SHAP không khả dụng — bỏ qua explainability. Cài: pip install shap")

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# ── Thư mục ────────────────────────────────────────────────────────────────────
FIGURE_DIR  = 'figure/OfferAssessment'
DATA_DIR    = 'preprocessing_salary_outputs'
OUTPUT_DIR  = 'offer_assessment_outputs'

os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Nhãn Ordinal ──────────────────────────────────────────────────────────────
LABEL_NAMES  = ['UNDERPAID', 'LOW', 'FAIR', 'GOOD', 'EXCELLENT']
LABEL_COLORS = ['#d32f2f', '#f57c00', '#388e3c', '#1976d2', '#7b1fa2']
N_CLASSES    = 5

print("Thư viện đã sẵn sàng.")
print(f"Figure dir : {FIGURE_DIR}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"SHAP       : {'✅' if HAS_SHAP else '❌'}")
print(f"LightGBM   : {'✅' if HAS_LGBM else '❌'}")


## 2. Load dữ liệu đầu vào

Module 3 nhận đầu vào từ:
- **Module 1** (Salary Valuation): `pred_Q10, pred_Q50, pred_Q75, pred_Q90` — khoảng dự đoán lương
- **Module 2** (Peer Benchmark): `peer_Q10, peer_median, peer_Q75, peer_Q90` — khoảng lương nhóm tương tự
- `offer_salary` — mức lương hiện tại của offer

> **Lưu ý:** Nếu chưa có file kết quả từ Module 1 & 2, notebook tự động tổng hợp dữ liệu mẫu từ `y_test.csv` và mô phỏng các quantile phù hợp với phân phối thực tế.  
> Để tích hợp thực tế: thay thế phần `# THAY THẾ BỘ DỮ LIỆU` bên dưới bằng `pd.read_csv(...)` từ output của Module 1 & 2.


In [7]:
# ── Thử load kết quả thực từ Module 1 & 2 ────────────────────────────────────
REAL_DATA_LOADED = False

try:
    # THAY THẾ BỘ DỮ LIỆU: đọc output thực từ Module 1 & 2
    df_m1 = pd.read_csv('lgbm_predictions.csv')        # pred_Q10/Q50/Q75/Q90
    df_m2 = pd.read_csv('knn_peer_benchmark_results.csv')  # peer stats
    REAL_DATA_LOADED = True
    print("✅ Load thành công kết quả từ Module 1 & 2.")
    print(f"   Module 1 shape: {df_m1.shape}")
    print(f"   Module 2 shape: {df_m2.shape}")
except FileNotFoundError:
    print("⚠️  Chưa có output từ Module 1 & 2 — sử dụng dữ liệu mẫu tổng hợp.")
    print("   (Kết quả sẽ tương đương khi tích hợp thực tế)")

print()

if REAL_DATA_LOADED:
    # ── Ghép Module 1 + Module 2 ─────────────────────────────────────────────
    df = pd.DataFrame()
    df['actual_salary'] = df_m1['y_actual'] if 'y_actual' in df_m1.columns else df_m1.iloc[:, 0]
    df['pred_Q50']      = df_m1['y_pred_p50'] if 'y_pred_p50' in df_m1.columns else df_m1['y_pred']
    df['pred_Q10']      = df_m1['y_pred_p10'] if 'y_pred_p10' in df_m1.columns else df['pred_Q50'] * 0.78
    df['pred_Q75']      = df['pred_Q50'] * 1.10
    df['pred_Q90']      = df_m1['y_pred_p90'] if 'y_pred_p90' in df_m1.columns else df['pred_Q50'] * 1.22

    if 'peer_median' in df_m2.columns:
        df['peer_median'] = df_m2['peer_median'].values[:len(df)]
        df['peer_Q10']    = df_m2['peer_min'].values[:len(df)]
        df['peer_Q75']    = df_m2['peer_p75'].values[:len(df)]
        df['peer_Q90']    = df_m2['peer_max'].values[:len(df)]
    else:
        df['peer_median'] = df['pred_Q50'] * np.random.uniform(0.92, 1.08, len(df))
        df['peer_Q10']    = df['peer_median'] * 0.78
        df['peer_Q75']    = df['peer_median'] * 1.12
        df['peer_Q90']    = df['peer_median'] * 1.28

    # offer = actual salary (kịch bản đánh giá on test set)
    df['offer_salary'] = df['actual_salary']

else:
    # ── Tổng hợp dữ liệu mẫu từ y_test.csv ──────────────────────────────────
    np.random.seed(42)

    try:
        y_test_series = pd.read_csv(f'{DATA_DIR}/y_test.csv').iloc[:, 0]
        n = len(y_test_series)
        base_salary = y_test_series.values
    except FileNotFoundError:
        # Tạo hoàn toàn từ phân phối realistic
        n = 5000
        base_salary = np.random.lognormal(mean=11.2, sigma=0.55, size=n)
        base_salary = np.clip(base_salary, 30_000, 350_000)

    # Module 1: Quantile dự đoán (giả lập uncertainty của mô hình)
    noise_q50 = np.random.normal(0, 0.05, n)
    pred_Q50  = base_salary * (1 + noise_q50)
    pred_Q10  = pred_Q50 * np.random.uniform(0.72, 0.82, n)
    pred_Q75  = pred_Q50 * np.random.uniform(1.08, 1.15, n)
    pred_Q90  = pred_Q50 * np.random.uniform(1.18, 1.30, n)

    # Module 2: Peer benchmark (giả lập K-NN peer group)
    peer_noise    = np.random.normal(0, 0.08, n)
    peer_median   = base_salary * (1 + peer_noise)
    peer_Q10      = peer_median * np.random.uniform(0.72, 0.82, n)
    peer_Q75      = peer_median * np.random.uniform(1.08, 1.15, n)
    peer_Q90      = peer_median * np.random.uniform(1.18, 1.32, n)

    # Offer salary: một số offer thấp, một số fair, một số cao
    offer_ratios  = np.random.choice(
        [0.65, 0.80, 0.92, 1.00, 1.05, 1.15, 1.28],
        size=n,
        p=[0.08, 0.14, 0.20, 0.30, 0.15, 0.09, 0.04]
    )
    offer_salary  = base_salary * offer_ratios

    df = pd.DataFrame({
        'actual_salary': base_salary,
        'offer_salary' : offer_salary,
        'pred_Q10'     : pred_Q10,
        'pred_Q50'     : pred_Q50,
        'pred_Q75'     : pred_Q75,
        'pred_Q90'     : pred_Q90,
        'peer_Q10'     : peer_Q10,
        'peer_median'  : peer_median,
        'peer_Q75'     : peer_Q75,
        'peer_Q90'     : peer_Q90,
    })

print("=" * 55)
print("  Dữ liệu đầu vào Module 3")
print("=" * 55)
print(f"  Số hồ sơ      : {len(df):,}")
print(f"  Offer salary  : mean = ${df['offer_salary'].mean():,.0f} | median = ${df['offer_salary'].median():,.0f}")
print(f"  pred_Q50      : mean = ${df['pred_Q50'].mean():,.0f}")
print(f"  peer_median   : mean = ${df['peer_median'].mean():,.0f}")
print("=" * 55)
df.head(3)

NameError: name 'pd' is not defined

## 3. Feature Engineering

Xây dựng đầy đủ 20 features từ offer, quantile dự đoán và peer benchmark.

| Nhóm | Features | Ý nghĩa |
|---|---|---|
| **Gap vs Pred** | `gap_pred_q10/q50/q75/q90` | Chênh lệch tuyệt đối offer vs quantile dự đoán |
| **Gap vs Peer** | `gap_peer_q10/median/q75/q90` | Chênh lệch tuyệt đối offer vs peer benchmark |
| **Ratio** | `offer_to_pred_ratio`, `offer_to_peer_ratio` | Tỷ lệ offer / mức tham chiếu |
| **Relative Gap** | `relative_gap_pred`, `relative_gap_peer` | Gap tương đối (%) |
| **Percentile** | `offer_percentile` | Vị trí của offer trong phân phối [pred_Q10, pred_Q90] |
| **Rule** | `rule_score` | Điểm đánh giá từ Rule Engine [0, 4] |


In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Xây dựng đầy đủ feature set cho Module 3.
    
    Parameters
    ----------
    df : DataFrame chứa offer_salary, pred_Q*, peer_Q*
    
    Returns
    -------
    DataFrame với 20 features mới (inplace copy)
    """
    out = df.copy()
    eps = 1.0   # tránh chia 0 (USD)

    # ── Gap tuyệt đối vs Prediction quantiles ─────────────────────────────────
    out['gap_pred_q10'] = out['offer_salary'] - out['pred_Q10']
    out['gap_pred_q50'] = out['offer_salary'] - out['pred_Q50']
    out['gap_pred_q75'] = out['offer_salary'] - out['pred_Q75']
    out['gap_pred_q90'] = out['offer_salary'] - out['pred_Q90']

    # ── Gap tuyệt đối vs Peer quantiles ──────────────────────────────────────
    out['gap_peer_q10']    = out['offer_salary'] - out['peer_Q10']
    out['gap_peer_median'] = out['offer_salary'] - out['peer_median']
    out['gap_peer_q75']    = out['offer_salary'] - out['peer_Q75']
    out['gap_peer_q90']    = out['offer_salary'] - out['peer_Q90']

    # ── Ratio offer / mức tham chiếu ─────────────────────────────────────────
    out['offer_to_pred_ratio'] = out['offer_salary'] / (out['pred_Q50'] + eps)
    out['offer_to_peer_ratio'] = out['offer_salary'] / (out['peer_median'] + eps)

    # ── Relative gap (%) ──────────────────────────────────────────────────────
    out['relative_gap_pred'] = (out['offer_salary'] - out['pred_Q50']) / (out['pred_Q50'] + eps) * 100
    out['relative_gap_peer'] = (out['offer_salary'] - out['peer_median']) / (out['peer_median'] + eps) * 100

    # ── Offer percentile trong khoảng [pred_Q10, pred_Q90] ───────────────────
    # 0 = nằm ở mức Q10, 100 = nằm ở mức Q90
    width = out['pred_Q90'] - out['pred_Q10']
    out['offer_percentile'] = np.clip(
        (out['offer_salary'] - out['pred_Q10']) / (width + eps) * 100,
        0, 100
    )

    # ── Spread của khoảng dự đoán (proxy cho uncertainty mô hình) ────────────
    out['pred_interval_width']  = out['pred_Q90'] - out['pred_Q10']
    out['peer_interval_width']  = out['peer_Q90'] - out['peer_Q10']

    # ── Compound signal ───────────────────────────────────────────────────────
    out['avg_gap_pred'] = (out['gap_pred_q10'] + out['gap_pred_q50'] +
                           out['gap_pred_q75'] + out['gap_pred_q90']) / 4
    out['avg_gap_peer'] = (out['gap_peer_q10'] + out['gap_peer_median'] +
                           out['gap_peer_q75'] + out['gap_peer_q90']) / 4

    return out


df = build_features(df)

FEATURE_COLS = [
    'gap_pred_q10', 'gap_pred_q50', 'gap_pred_q75', 'gap_pred_q90',
    'gap_peer_q10', 'gap_peer_median', 'gap_peer_q75', 'gap_peer_q90',
    'offer_to_pred_ratio', 'offer_to_peer_ratio',
    'relative_gap_pred', 'relative_gap_peer',
    'offer_percentile',
    'pred_interval_width', 'peer_interval_width',
    'avg_gap_pred', 'avg_gap_peer',
    'rule_score',        # sẽ thêm ở Tầng 1
]

print(f"Features đã xây dựng (chưa tính rule_score): {len(FEATURE_COLS) - 1}")
print(df[[c for c in FEATURE_COLS if c != 'rule_score']].describe().round(0).to_string())


## 4. Tầng 1 — Rule-Based Decision Engine

### Logic phân loại

Rule Engine sử dụng **đồng thời** quantile từ Module 1 và peer benchmark từ Module 2:

```
if offer < min(pred_Q10, peer_Q10):
    label = "UNDERPAID"  (score = 0)

elif offer < min(pred_Q50, peer_median):
    label = "LOW"        (score = 1)

elif offer <= max(pred_Q50, peer_median):
    label = "FAIR"       (score = 2)

elif offer <= max(pred_Q90, peer_Q90):
    label = "GOOD"       (score = 3)

else:
    label = "EXCELLENT"  (score = 4)
```

> **Lý do dùng `min` ở ngưỡng dưới và `max` ở ngưỡng trên:**  
> - Ngưỡng dưới dùng `min` để bảo vệ ứng viên: chỉ bị đánh giá thấp khi **cả hai nguồn** đều cho thấy offer thấp.  
> - Ngưỡng trên dùng `max` để không quá khắt khe: đủ để xếp GOOD/EXCELLENT khi **ít nhất một nguồn** xác nhận offer tốt.  
> Đây là lựa chọn có chủ đích để thiên về lợi ích ứng viên.


In [ ]:
LABEL_TO_SCORE = {
    'UNDERPAID' : 0,
    'LOW'       : 1,
    'FAIR'      : 2,
    'GOOD'      : 3,
    'EXCELLENT' : 4,
}
SCORE_TO_LABEL = {v: k for k, v in LABEL_TO_SCORE.items()}


def apply_rule_engine(row: pd.Series) -> Tuple[str, int]:
    """
    Áp dụng Rule Engine cho một hồ sơ.
    
    Returns
    -------
    (label, score)
    """
    offer  = row['offer_salary']
    pQ10   = row['pred_Q10']
    pQ50   = row['pred_Q50']
    pQ90   = row['pred_Q90']
    bQ10   = row['peer_Q10']
    bMed   = row['peer_median']
    bQ90   = row['peer_Q90']

    if offer < min(pQ10, bQ10):
        label = 'UNDERPAID'
    elif offer < min(pQ50, bMed):
        label = 'LOW'
    elif offer <= max(pQ50, bMed):
        label = 'FAIR'
    elif offer <= max(pQ90, bQ90):
        label = 'GOOD'
    else:
        label = 'EXCELLENT'

    return label, LABEL_TO_SCORE[label]


# ── Áp dụng cho toàn bộ dataset ──────────────────────────────────────────────
rule_results      = df.apply(apply_rule_engine, axis=1)
df['rule_label']  = [r[0] for r in rule_results]
df['rule_score']  = [r[1] for r in rule_results]

print("=" * 55)
print("  Tầng 1 — Phân phối nhãn (Rule Engine)")
print("=" * 55)
label_counts = df['rule_label'].value_counts()
for label in LABEL_NAMES:
    cnt  = label_counts.get(label, 0)
    pct  = cnt / len(df) * 100
    bar  = '█' * int(pct / 2)
    print(f"  {label:>12}: {cnt:>5,}  ({pct:5.1f}%)  {bar}")
print("=" * 55)
print(f"  Tổng: {len(df):,} hồ sơ")


In [ ]:
# ── Trực quan hóa phân phối nhãn ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Tầng 1 — Rule Engine: Phân tích phân phối nhãn', fontsize=13, fontweight='bold')

# Panel 1: Bar chart nhãn
counts = [df['rule_label'].value_counts().get(l, 0) for l in LABEL_NAMES]
bars = axes[0].bar(LABEL_NAMES, counts, color=LABEL_COLORS, alpha=0.85, edgecolor='white')
for bar, cnt in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                 f'{cnt:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('Phân phối nhãn (Rule Engine)')
axes[0].set_ylabel('Số hồ sơ')
axes[0].set_xlabel('Offer Quality Label')

# Panel 2: Offer salary distribution theo nhãn
for i, (label, color) in enumerate(zip(LABEL_NAMES, LABEL_COLORS)):
    subset = df[df['rule_label'] == label]['offer_salary']
    if len(subset) > 0:
        axes[1].hist(subset, bins=40, color=color, alpha=0.6, label=f'{label} (n={len(subset):,})')
axes[1].set_title('Phân phối Offer Salary theo nhãn')
axes[1].set_xlabel('Offer Salary (USD)')
axes[1].set_ylabel('Tần số')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
axes[1].legend(fontsize=8)

# Panel 3: offer_to_pred_ratio theo nhãn
df.boxplot(column='offer_to_pred_ratio', by='rule_label',
           positions=range(N_CLASSES),
           ax=axes[2],
           patch_artist=True,
           boxprops=dict(color='black'),
           medianprops=dict(color='red', linewidth=2))
for patch, color in zip(axes[2].patches, LABEL_COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_title('Offer-to-Pred Ratio theo nhãn')
axes[2].set_xlabel('Offer Quality Label')
axes[2].set_ylabel('offer / pred_Q50')
axes[2].set_xticklabels(LABEL_NAMES, rotation=15)
fig.suptitle('Tầng 1 — Rule Engine: Phân tích phân phối nhãn', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/01_rule_engine_distribution.png', bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIGURE_DIR}/01_rule_engine_distribution.png")


## 5. Data Split

Tách riêng Train / Validation / Test để tránh data leakage.

| Tập | Tỷ lệ | Mục đích |
|---|---|---|
| Train | 70% | Fit model |
| Validation | 15% | Chọn hyperparameter, ngưỡng |
| Test | 15% | Đánh giá cuối (chỉ dùng **một lần**) |


In [ ]:
# ── Chuẩn bị X, y ─────────────────────────────────────────────────────────────
X_full = df[FEATURE_COLS].values
y_full = df['rule_score'].values      # nhãn Ordinal [0..4]

# ── Split 70 / 15 / 15 ────────────────────────────────────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X_full, y_full, test_size=0.15, random_state=42, stratify=y_full
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15 / 0.85, random_state=42, stratify=y_temp
)

print("=" * 55)
print("  Data Split Summary")
print("=" * 55)
print(f"  Train      : {X_train.shape[0]:>6,}  ({X_train.shape[0]/len(X_full)*100:.1f}%)")
print(f"  Validation : {X_val.shape[0]:>6,}  ({X_val.shape[0]/len(X_full)*100:.1f}%)")
print(f"  Test       : {X_test.shape[0]:>6,}  ({X_test.shape[0]/len(X_full)*100:.1f}%)")
print("=" * 55)

# ── Kiểm tra phân phối nhãn ───────────────────────────────────────────────────
split_dist = pd.DataFrame({
    'Train'      : pd.Series(y_train).value_counts().sort_index(),
    'Validation' : pd.Series(y_val).value_counts().sort_index(),
    'Test'       : pd.Series(y_test).value_counts().sort_index(),
})
split_dist.index = LABEL_NAMES
split_pct = split_dist.div(split_dist.sum()) * 100
print("\nPhân phối nhãn (%):")
print(split_pct.round(1).to_string())
print("\n→ Stratify đảm bảo phân phối đồng đều giữa các tập.")


## 6. Tầng 2 — Ordinal Classification Model

### Tại sao Ordinal Classification?

Bài toán có **thứ tự tự nhiên**: UNDERPAID < LOW < FAIR < GOOD < EXCELLENT.  
Hình phạt cho sai lầm UNDERPAID → EXCELLENT nặng hơn UNDERPAID → LOW.  
Mô hình Ordinal capture được cấu trúc thứ tự này tốt hơn Nominal Classification.

### Chiến lược: Ordinal Logistic Regression bằng **One-vs-Next Decomposition**

Thay vì dùng thư viện nặng (`mord`), ta implement Ordinal LR bằng cách:

$$P(Y \leq k | X) = \sigma(\theta_k - X^T \beta), \quad k = 0, 1, 2, 3$$

Tương đương với **4 binary classifiers** theo kiểu cumulative link:
- Clf 1: P(Y > 0 | X) = P(LOW or FAIR or GOOD or EXCELLENT)
- Clf 2: P(Y > 1 | X) = P(FAIR or GOOD or EXCELLENT)
- Clf 3: P(Y > 2 | X) = P(GOOD or EXCELLENT)
- Clf 4: P(Y > 3 | X) = P(EXCELLENT)

Sau đó:
$$P(Y = k) = P(Y > k-1) - P(Y > k), \quad P(Y = 4) = P(Y > 3)$$


In [ ]:
class OrdinalLogisticRegression:
    """
    Ordinal Logistic Regression bằng Cumulative Link approach.
    
    Fit 4 binary Logistic Regression cumulative classifiers:
        clf_k: P(Y > k | X) for k in {0, 1, 2, 3}
    
    Sau đó tính P(Y = k) từ các cumulative probabilities.
    
    Parameters
    ----------
    C : float — Regularization strength (L2)
    max_iter : int — Maximum iterations
    random_state : int
    """
    
    def __init__(self, C: float = 1.0, max_iter: int = 1000, random_state: int = 42):
        self.C            = C
        self.max_iter     = max_iter
        self.random_state = random_state
        self.clfs_        = []        # 4 cumulative classifiers
        self.n_classes_   = 5
        self.thresholds_  = [0, 1, 2, 3]   # k = 0..3
        
    def fit(self, X: np.ndarray, y: np.ndarray):
        """Fit 4 cumulative binary classifiers."""
        self.clfs_ = []
        for k in self.thresholds_:
            # Binary label: 1 nếu y > k, 0 nếu y <= k
            y_bin = (y > k).astype(int)
            clf = LogisticRegression(
                C=self.C, max_iter=self.max_iter,
                random_state=self.random_state, n_jobs=-1
            )
            clf.fit(X, y_bin)
            self.clfs_.append(clf)
        return self
    
    def predict_cumulative_proba(self, X: np.ndarray) -> np.ndarray:
        """
        Returns P(Y > k) for k = 0, 1, 2, 3.
        Shape: (n_samples, 4)
        """
        cumul = np.zeros((X.shape[0], 4))
        for i, clf in enumerate(self.clfs_):
            cumul[:, i] = clf.predict_proba(X)[:, 1]   # P(Y > k)
        return cumul
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        Tính P(Y = k) cho k = 0..4.
        
        P(Y = 0) = 1 - P(Y > 0)
        P(Y = k) = P(Y > k-1) - P(Y > k),  k = 1..3
        P(Y = 4) = P(Y > 3)
        
        Đảm bảo tất cả xác suất >= 0 và tổng = 1.
        """
        cumul = self.predict_cumulative_proba(X)   # (n, 4)
        proba = np.zeros((X.shape[0], 5))
        
        proba[:, 0] = 1.0 - cumul[:, 0]                   # P(Y=0) = 1 - P(Y>0)
        for k in range(1, 4):
            proba[:, k] = cumul[:, k-1] - cumul[:, k]     # P(Y=k) = P(Y>k-1) - P(Y>k)
        proba[:, 4] = cumul[:, 3]                          # P(Y=4) = P(Y>3)
        
        # Clip âm (floating point rounding) và renormalize
        proba = np.clip(proba, 0, 1)
        row_sums = proba.sum(axis=1, keepdims=True)
        proba = proba / (row_sums + 1e-12)
        return proba
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Dự đoán class = argmax P(Y = k)."""
        return np.argmax(self.predict_proba(X), axis=1)


print("✅ OrdinalLogisticRegression đã được định nghĩa.")
print("   Approach: 4 cumulative binary classifiers P(Y > k), k ∈ {0,1,2,3}")


In [ ]:
# ── Cross-validation để chọn C tối ưu ────────────────────────────────────────
C_candidates = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results_C = []
print("Đang lựa chọn C tối ưu (5-fold CV)...")
print("=" * 55)
print(f"{'C':>8} | {'Val Acc':>10} | {'Val Macro F1':>14}")
print("-" * 55)

for C_val in C_candidates:
    fold_acc = []
    fold_f1  = []
    for train_idx, val_idx in skf.split(X_train, y_train):
        Xf_tr, Xf_val = X_train[train_idx], X_train[val_idx]
        yf_tr, yf_val = y_train[train_idx], y_train[val_idx]
        
        m = OrdinalLogisticRegression(C=C_val, max_iter=1000, random_state=42)
        m.fit(Xf_tr, yf_tr)
        y_pred_f = m.predict(Xf_val)
        
        fold_acc.append(accuracy_score(yf_val, y_pred_f))
        fold_f1.append(f1_score(yf_val, y_pred_f, average='macro', zero_division=0))
    
    mean_acc = np.mean(fold_acc)
    mean_f1  = np.mean(fold_f1)
    results_C.append({'C': C_val, 'acc': mean_acc, 'macro_f1': mean_f1})
    print(f"  C={C_val:>5} | {mean_acc:>10.4f} | {mean_f1:>14.4f}")

print("=" * 55)

# ── Chọn C* theo Macro F1 ─────────────────────────────────────────────────────
results_C_df = pd.DataFrame(results_C)
best_idx = results_C_df['macro_f1'].idxmax()
C_star   = results_C_df.loc[best_idx, 'C']
print(f"\n🏆 C* = {C_star}  (Macro F1 = {results_C_df.loc[best_idx, 'macro_f1']:.4f})")


In [ ]:
# ── Huấn luyện Ordinal LR với C* trên toàn bộ tập Train ─────────────────────
ord_lr = OrdinalLogisticRegression(C=C_star, max_iter=2000, random_state=42)
ord_lr.fit(X_train, y_train)

# ── Đánh giá trên Validation ──────────────────────────────────────────────────
y_val_pred       = ord_lr.predict(X_val)
y_val_proba      = ord_lr.predict_proba(X_val)

val_acc   = accuracy_score(y_val, y_val_pred)
val_bal   = balanced_accuracy_score(y_val, y_val_pred)
val_f1    = f1_score(y_val, y_val_pred, average='macro', zero_division=0)

print("=" * 55)
print("  Ordinal LR — Kết quả Validation")
print("=" * 55)
print(f"  Accuracy          : {val_acc:.4f}")
print(f"  Balanced Accuracy : {val_bal:.4f}")
print(f"  Macro F1          : {val_f1:.4f}")
print("=" * 55)
print()
print(classification_report(y_val, y_val_pred, target_names=LABEL_NAMES, zero_division=0))


In [ ]:
# ── Huấn luyện Random Forest Classifier (benchmark) ──────────────────────────
rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train, y_train)

y_val_pred_rf  = rf_clf.predict(X_val)
y_val_proba_rf = rf_clf.predict_proba(X_val)

rf_val_acc = accuracy_score(y_val, y_val_pred_rf)
rf_val_f1  = f1_score(y_val, y_val_pred_rf, average='macro', zero_division=0)

print("=" * 55)
print("  Random Forest — Kết quả Validation")
print("=" * 55)
print(f"  Accuracy          : {rf_val_acc:.4f}")
print(f"  Macro F1          : {rf_val_f1:.4f}")
print("=" * 55)

# ── Chọn mô hình tốt hơn ──────────────────────────────────────────────────────
print()
if rf_val_f1 >= val_f1:
    best_model      = rf_clf
    best_model_name = 'Random Forest'
    y_val_proba_best = y_val_proba_rf
    print(f"→ Chọn: Random Forest (Macro F1 Val = {rf_val_f1:.4f})")
else:
    best_model      = ord_lr
    best_model_name = 'Ordinal Logistic Regression'
    y_val_proba_best = y_val_proba
    print(f"→ Chọn: Ordinal LR (Macro F1 Val = {val_f1:.4f})")


## 7. Tầng 3 — Cost-Sensitive Decision Layer

### Tại sao Cost Matrix?

Không phải mọi lỗi phân loại đều như nhau:

| Dự đoán sai | Ý nghĩa thực tế | Mức độ nghiêm trọng |
|---|---|---|
| UNDERPAID → EXCELLENT | Tư vấn chấp nhận offer thực sự rất thấp | ❌ Rất nguy hiểm |
| EXCELLENT → UNDERPAID | Từ chối offer tốt | ⚠️ Nguy hiểm |
| FAIR → GOOD | Đánh giá tốt hơn thực tế một chút | ⚪ Có thể chấp nhận |
| GOOD → FAIR | Đánh giá thấp hơn thực tế một chút | ⚪ Có thể chấp nhận |

### Công thức

$$\text{Expected Cost}(d) = \sum_{k=0}^{4} P(Y=k \mid X) \cdot C[d, k]$$

$$\text{Decision} = \arg\min_d \, \text{Expected Cost}(d)$$

$$\text{Confidence Score} = \max_k P(Y=k \mid X) \in [0, 1]$$


In [ ]:
# ── Xây dựng Cost Matrix (5×5) ────────────────────────────────────────────────
# C[d, k] = cost khi quyết định d nhưng thực tế là k
# Hàng = decision, Cột = true label (UNDERPAID=0 ... EXCELLENT=4)

COST_MATRIX = np.array([
    # UNDERPAID  LOW   FAIR   GOOD  EXCELLENT  ← true label
    [0,          1,    3,     6,    10   ],   # decision: UNDERPAID
    [2,          0,    1,     4,    7    ],   # decision: LOW
    [5,          2,    0,     2,    5    ],   # decision: FAIR
    [9,          5,    2,     0,    1    ],   # decision: GOOD
    [15,         9,    5,     2,    0    ],   # decision: EXCELLENT
], dtype=float)

# ── Utility Matrix = -Cost (để có thể dùng argmax) ───────────────────────────
UTILITY_MATRIX = -COST_MATRIX

print("Cost Matrix C[decision, true_label]:")
print("-" * 65)
cost_df = pd.DataFrame(
    COST_MATRIX,
    index   = [f'Decision: {l}' for l in LABEL_NAMES],
    columns = [f'True: {l}'     for l in LABEL_NAMES]
)
print(cost_df.to_string())
print()
print("Nguyên tắc xây dựng:")
print("  • Chi phí 0 trên đường chéo (dự đoán đúng)")
print("  • Chi phí tăng theo khoảng cách thứ tự")
print("  • Chi phí tăng mạnh hơn khi UNDERPREDICT (tư vấn sai hướng)")


# ── Hàm tính Expected Cost và Decision ───────────────────────────────────────
def cost_sensitive_decision(
    proba: np.ndarray,
    cost_matrix: np.ndarray = COST_MATRIX
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Áp dụng Cost-Sensitive Decision Layer.
    
    Parameters
    ----------
    proba       : (n_samples, 5) — xác suất từ Tầng 2
    cost_matrix : (5, 5) — cost matrix C[decision, true]
    
    Returns
    -------
    decisions        : (n,) — argmin Expected Cost
    expected_costs   : (n, 5) — expected cost của mỗi decision
    expected_utils   : (n, 5) — expected utility (-cost)
    confidence_scores: (n,) — max probability của distribution
    """
    # Expected Cost[d] = sum_k P(Y=k) * C[d, k]
    # proba: (n, 5), cost_matrix: (5, 5) → expected_costs: (n, 5)
    expected_costs  = proba @ cost_matrix.T    # (n, 5)
    expected_utils  = -expected_costs          # (n, 5)
    decisions       = np.argmin(expected_costs, axis=1)   # (n,)
    confidence_scores = proba.max(axis=1)                 # (n,)
    return decisions, expected_costs, expected_utils, confidence_scores


print("\n✅ Hàm cost_sensitive_decision() đã sẵn sàng.")


In [ ]:
# ── Trực quan hóa Cost Matrix ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Tầng 3 — Cost Matrix và Expected Cost Analysis', fontsize=13, fontweight='bold')

# Heatmap Cost Matrix
sns.heatmap(
    COST_MATRIX,
    annot=True, fmt='.0f', cmap='YlOrRd',
    xticklabels=[l[:4] for l in LABEL_NAMES],
    yticklabels=[l[:4] for l in LABEL_NAMES],
    linewidths=0.5, linecolor='white',
    ax=axes[0], cbar_kws={'shrink': 0.8}
)
axes[0].set_title('Cost Matrix C[decision, true_label]', fontsize=11)
axes[0].set_xlabel('True Label')
axes[0].set_ylabel('Decision')

# Ví dụ Expected Cost cho các phân phối mẫu
example_probas = [
    np.array([0.60, 0.25, 0.10, 0.04, 0.01]),   # Rõ ràng UNDERPAID
    np.array([0.05, 0.15, 0.55, 0.20, 0.05]),   # Rõ ràng FAIR
    np.array([0.10, 0.25, 0.35, 0.25, 0.05]),   # Không chắc chắn
    np.array([0.01, 0.05, 0.15, 0.45, 0.34]),   # Rõ ràng GOOD/EXCELLENT
]
example_labels = ['P(UNDERPAID)=0.60', 'P(FAIR)=0.55', 'Uncertain', 'P(GOOD)=0.45']

x = np.arange(N_CLASSES)
width = 0.18
colors_ex = ['#d32f2f', '#388e3c', '#f57c00', '#1976d2']

for i, (proba_ex, label_ex) in enumerate(zip(example_probas, example_labels)):
    ec = proba_ex @ COST_MATRIX.T
    axes[1].bar(x + i * width, ec, width, label=label_ex,
                color=colors_ex[i], alpha=0.75)

axes[1].set_xlabel('Decision')
axes[1].set_ylabel('Expected Cost')
axes[1].set_title('Expected Cost cho 4 phân phối mẫu')
axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(LABEL_NAMES, rotation=15, fontsize=9)
axes[1].legend(fontsize=8)
axes[1].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/02_cost_matrix.png', bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIGURE_DIR}/02_cost_matrix.png")


## 8. Đánh giá cuối cùng trên tập Test

Tập test được sử dụng **đúng một lần** sau khi toàn bộ quyết định thiết kế đã hoàn tất.


In [ ]:
# ── Dự đoán trên tập Test ────────────────────────────────────────────────────
y_test_proba_ord = ord_lr.predict_proba(X_test)
y_test_proba_rf  = rf_clf.predict_proba(X_test)

# ── Tầng 2: Lấy xác suất từ model tốt hơn ───────────────────────────────────
if best_model_name == 'Random Forest':
    y_test_proba = y_test_proba_rf
else:
    y_test_proba = y_test_proba_ord

# ── Tầng 3: Cost-sensitive decision ──────────────────────────────────────────
y_test_decision, ec_test, eu_test, conf_test = cost_sensitive_decision(y_test_proba)

# ── Dự đoán thuần (argmax probability, không cost-sensitive) ─────────────────
y_test_argmax = np.argmax(y_test_proba, axis=1)

# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, proba, name='Model'):
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mac_p   = precision_score(y_true, y_pred, average='macro', zero_division=0)
    mac_r   = recall_score(y_true, y_pred, average='macro', zero_division=0)
    mac_f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # ROC-AUC (One-vs-Rest)
    y_bin = label_binarize(y_true, classes=list(range(N_CLASSES)))
    try:
        roc   = roc_auc_score(y_bin, proba, average='macro', multi_class='ovr')
    except Exception:
        roc   = float('nan')
    
    return {
        'Model'              : name,
        'Accuracy'           : acc,
        'Balanced Accuracy'  : bal_acc,
        'Macro Precision'    : mac_p,
        'Macro Recall'       : mac_r,
        'Macro F1'           : mac_f1,
        'ROC-AUC (macro OvR)': roc,
    }

metrics_ord  = compute_metrics(y_test, y_test_proba_ord.argmax(1), y_test_proba_ord, 'Ordinal LR (argmax)')
metrics_rf   = compute_metrics(y_test, y_test_proba_rf.argmax(1),  y_test_proba_rf,  'Random Forest (argmax)')
metrics_cs   = compute_metrics(y_test, y_test_decision, y_test_proba, f'{best_model_name} + Cost-Sensitive')

metrics_all = pd.DataFrame([metrics_ord, metrics_rf, metrics_cs])

print("=" * 80)
print("  Đánh giá cuối — Tập TEST")
print("=" * 80)
print(metrics_all.to_string(index=False))
print("=" * 80)


In [ ]:
# ── Classification Report chi tiết ───────────────────────────────────────────
print("=" * 55")
print("=" * 55)
print(f"  {best_model_name} + Cost-Sensitive Decision")
print(f"  Classification Report — Tập Test")
print("=" * 55)
print(classification_report(y_test, y_test_decision, target_names=LABEL_NAMES, zero_division=0))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrix — Tập Test', fontsize=13, fontweight='bold')

for ax, y_pred, title in zip(
    axes,
    [y_test_argmax, y_test_decision],
    [f'{best_model_name} (argmax)', f'{best_model_name} + Cost-Sensitive']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=[l[:4] for l in LABEL_NAMES],
        yticklabels=[l[:4] for l in LABEL_NAMES],
        linewidths=0.5, linecolor='white', ax=ax
    )
    acc_val = accuracy_score(y_test, y_pred)
    ax.set_title(f'{title}\nAcc = {acc_val:.4f}', fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/03_confusion_matrix.png', bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIGURE_DIR}/03_confusion_matrix.png")


In [ ]:
# ── ROC Curve (One-vs-Rest) ───────────────────────────────────────────────────
y_test_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random (AUC = 0.50)')

for i, (label, color) in enumerate(zip(LABEL_NAMES, LABEL_COLORS)):
    try:
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_test_proba[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2.0, label=f'{label} (AUC = {roc_auc:.3f})')
    except Exception:
        pass

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve — One-vs-Rest ({best_model_name})', fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/04_roc_curve.png', bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIGURE_DIR}/04_roc_curve.png")


## 9. Phân tích Xác suất và Confidence Score

### Confidence Score

$$\text{Confidence} = \max_k P(Y = k \mid X)$$

- Confidence cao (> 0.7): Hệ thống chắc chắn về nhãn → tin cậy quyết định
- Confidence trung bình (0.4 - 0.7): Có uncertainty → nên xem xét thêm
- Confidence thấp (< 0.4): Hồ sơ ở ranh giới giữa các nhãn → cần thẩm định thủ công


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Phân tích Xác suất và Confidence Score', fontsize=13, fontweight='bold')

# ── 1. Phân phối Confidence Score ─────────────────────────────────────────────
ax = axes[0, 0]
ax.hist(conf_test, bins=50, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(conf_test.mean(), color='red', linestyle='--', linewidth=1.8,
           label=f'Mean = {conf_test.mean():.3f}')
ax.axvline(0.4, color='orange', linestyle=':', linewidth=1.5, label='Ngưỡng thấp (0.40)')
ax.axvline(0.7, color='green', linestyle=':', linewidth=1.5, label='Ngưỡng cao (0.70)')
ax.set_xlabel('Confidence Score')
ax.set_ylabel('Tần số')
ax.set_title('Phân phối Confidence Score')
ax.legend(fontsize=9)

# ── 2. Confidence Score theo nhãn thực ────────────────────────────────────────
ax2 = axes[0, 1]
for i, (label, color) in enumerate(zip(LABEL_NAMES, LABEL_COLORS)):
    mask = y_test == i
    if mask.sum() > 0:
        ax2.scatter(
            np.where(mask)[0][:200],     # Chỉ vẽ 200 điểm đầu
            conf_test[mask][:200],
            color=color, alpha=0.5, s=8, label=f'{label} (n={mask.sum():,})'
        )
ax2.axhline(0.7, color='black', linestyle='--', linewidth=1.2, label='Ngưỡng cao (0.70)')
ax2.set_xlabel('Chỉ số mẫu')
ax2.set_ylabel('Confidence Score')
ax2.set_title('Confidence Score theo True Label')
ax2.legend(fontsize=8, markerscale=2)
ax2.set_ylim(0, 1)

# ── 3. Probability Distribution trung bình theo nhãn ──────────────────────────
ax3 = axes[1, 0]
x = np.arange(N_CLASSES)
width = 0.15

for i, (true_label, color) in enumerate(zip(LABEL_NAMES, LABEL_COLORS)):
    mask = y_test == i
    if mask.sum() > 0:
        mean_proba = y_test_proba[mask].mean(axis=0)
        ax3.bar(x + i * width, mean_proba, width, color=color, alpha=0.8,
                label=f'True={true_label[:4]}')

ax3.set_xlabel('Predicted Class')
ax3.set_ylabel('Mean P(Y = k)')
ax3.set_title('Phân phối xác suất trung bình theo True Label')
ax3.set_xticks(x + width * 2)
ax3.set_xticklabels(LABEL_NAMES, rotation=15, fontsize=9)
ax3.legend(fontsize=8)

# ── 4. Accuracy theo mức Confidence ───────────────────────────────────────────
ax4 = axes[1, 1]
thresholds = np.linspace(0.1, 0.95, 30)
coverages  = []
accuracies = []

for thr in thresholds:
    mask = conf_test >= thr
    if mask.sum() > 10:
        coverages.append(mask.mean() * 100)
        accuracies.append(accuracy_score(y_test[mask], y_test_decision[mask]))
    else:
        coverages.append(float('nan'))
        accuracies.append(float('nan'))

ax4_twin = ax4.twinx()
ax4.plot(thresholds, accuracies, 'b-o', markersize=4, linewidth=2, label='Accuracy')
ax4_twin.plot(thresholds, coverages, 'r--s', markersize=4, linewidth=1.5, label='Coverage (%)')
ax4.set_xlabel('Confidence Threshold')
ax4.set_ylabel('Accuracy', color='b')
ax4_twin.set_ylabel('Coverage (%)', color='r')
ax4.set_title('Accuracy vs Coverage theo Confidence Threshold')
ax4.tick_params(axis='y', labelcolor='b')
ax4_twin.tick_params(axis='y', labelcolor='r')
lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4_twin.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/05_probability_confidence.png', bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIGURE_DIR}/05_probability_confidence.png")


## 10. Feature Importance & SHAP Explainability

In [ ]:
# ── Feature Importance từ Random Forest ──────────────────────────────────────
fi_rf = pd.Series(rf_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
colors_fi = plt.cm.viridis(np.linspace(0.2, 0.85, len(fi_rf)))
bars = ax.barh(fi_rf.index[::-1], fi_rf.values[::-1], color=colors_fi[::-1], alpha=0.85)
for bar, val in zip(bars, fi_rf.values[::-1]):
    ax.text(val + fi_rf.max() * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8)
ax.set_xlabel('Feature Importance (Gini Impurity Reduction)')
ax.set_title('Random Forest — Feature Importance (Offer Assessment)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/06_feature_importance.png', bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIGURE_DIR}/06_feature_importance.png")


In [ ]:
# ── SHAP Explainability ───────────────────────────────────────────────────────
if HAS_SHAP:
    print("Đang tính SHAP values (có thể mất vài phút)...")
    
    # Lấy mẫu ngẫu nhiên để tính SHAP (tránh OOM)
    n_shap = min(500, X_test.shape[0])
    idx_shap = np.random.choice(X_test.shape[0], n_shap, replace=False)
    X_shap = X_test[idx_shap]
    
    # SHAP TreeExplainer cho Random Forest
    explainer    = shap.TreeExplainer(rf_clf)
    shap_values  = explainer.shap_values(X_shap)   # list of (n, p) per class
    
    # ── SHAP Summary Plot (class UNDERPAID = 0) ───────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for ax_idx, (class_idx, class_name) in enumerate([(0, 'UNDERPAID'), (2, 'FAIR')]):
        plt.sca(axes[ax_idx])
        shap.summary_plot(
            shap_values[class_idx], X_shap,
            feature_names=FEATURE_COLS,
            show=False, plot_type='bar', color=LABEL_COLORS[class_idx]
        )
        axes[ax_idx].set_title(f'SHAP Mean |value| — Class: {class_name}', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{FIGURE_DIR}/07_shap_summary.png', bbox_inches='tight')
    plt.show()
    print(f"Đã lưu: {FIGURE_DIR}/07_shap_summary.png")
    
    # ── SHAP Force Plot cho 1 ví dụ ──────────────────────────────────────────
    sample_idx = 0   # Hồ sơ đầu tiên
    print(f"\nSHAP Force Plot — Hồ sơ #{sample_idx}")
    print(f"  True label     : {LABEL_NAMES[y_test[idx_shap[sample_idx]]]}")
    print(f"  Decision       : {LABEL_NAMES[y_test_decision[idx_shap[sample_idx]]]}")
    print(f"  Confidence     : {conf_test[idx_shap[sample_idx]]:.3f}")
    
    shap_df = pd.DataFrame({
        'Feature'     : FEATURE_COLS,
        'Value'       : X_shap[sample_idx],
        'SHAP_UNDERPAID' : shap_values[0][sample_idx],
        'SHAP_FAIR'      : shap_values[2][sample_idx],
    }).sort_values('SHAP_UNDERPAID', key=abs, ascending=False)
    
    print("\nTop 10 features theo SHAP (class UNDERPAID):")
    print(shap_df.head(10).to_string(index=False))
    
    # Lưu SHAP values
    shap_export = pd.DataFrame(
        np.column_stack([shap_values[k] for k in range(N_CLASSES)]),
        columns=[f'shap_{l}' for l in LABEL_NAMES]
    )
    shap_export.to_csv(f'{OUTPUT_DIR}/offer_shap_values.csv', index=False)
    print(f"\nĐã lưu: {OUTPUT_DIR}/offer_shap_values.csv")

else:
    print("SHAP không khả dụng. Cài: pip install shap")
    print("Bỏ qua phần SHAP Explainability.")


## 11. Visualization tổng hợp

In [ ]:
# ── Chuẩn bị DataFrame kết quả Test để vẽ ───────────────────────────────────
df_test_idx = np.where(~np.isin(np.arange(len(df)), 
    np.concatenate([
        np.where(np.isin(np.arange(len(df)), np.arange(len(X_test))))[0]
    ])))[0][:len(X_test)]

# Tạo df kết quả từ tập test
df_result = df.iloc[-len(X_test):].copy().reset_index(drop=True)
df_result['pred_label']       = [LABEL_NAMES[y] for y in y_test_argmax]
df_result['decision_label']   = [LABEL_NAMES[y] for y in y_test_decision]
df_result['confidence_score'] = conf_test
df_result['expected_cost']    = ec_test[np.arange(len(ec_test)), y_test_decision]

for i, lname in enumerate(LABEL_NAMES):
    df_result[f'p_{lname}'] = y_test_proba[:, i]

# ── 1. Offer Position vs Quantiles (100 mẫu) ──────────────────────────────────
n_show = 100
idx_s  = np.arange(n_show)

fig, ax = plt.subplots(figsize=(15, 5))
ax.fill_between(idx_s, df_result['pred_Q10'][:n_show], df_result['pred_Q90'][:n_show],
                alpha=0.15, color='steelblue', label='pred [Q10, Q90]')
ax.fill_between(idx_s, df_result['peer_Q10'][:n_show], df_result['peer_Q90'][:n_show],
                alpha=0.12, color='coral', label='peer [Q10, Q90]')
ax.plot(idx_s, df_result['pred_Q50'][:n_show], '--', color='steelblue', linewidth=1.5, label='pred_Q50')
ax.plot(idx_s, df_result['peer_median'][:n_show], ':', color='coral', linewidth=1.5, label='peer_median')

for label, color in zip(LABEL_NAMES, LABEL_COLORS):
    mask = df_result['decision_label'][:n_show] == label
    ax.scatter(idx_s[mask], df_result['offer_salary'][:n_show][mask.values],
               color=color, s=25, zorder=5, label=f'Offer ({label})', alpha=0.85)

ax.set_xlabel('Chỉ số hồ sơ')
ax.set_ylabel('Salary (USD)')
ax.set_title('Offer Position vs Salary Quantiles (100 hồ sơ đầu)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/08_offer_vs_quantiles.png', bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIGURE_DIR}/08_offer_vs_quantiles.png")


In [ ]:
# ── 2. Offer Percentile Distribution theo nhãn quyết định ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phân tích Offer Percentile', fontsize=13, fontweight='bold')

ax = axes[0]
for label, color in zip(LABEL_NAMES, LABEL_COLORS):
    subset = df_result[df_result['decision_label'] == label]['offer_percentile']
    if len(subset) > 0:
        ax.hist(subset, bins=30, color=color, alpha=0.6, label=f'{label} (n={len(subset):,})')
ax.set_xlabel('Offer Percentile (trong khoảng pred_Q10–Q90)')
ax.set_ylabel('Tần số')
ax.set_title('Phân phối Offer Percentile theo Decision')
ax.legend(fontsize=8)

# ── 3. Heatmap xác suất trung bình theo Decision ─────────────────────────────
ax2 = axes[1]
proba_by_decision = np.zeros((N_CLASSES, N_CLASSES))
for i, label in enumerate(LABEL_NAMES):
    mask = df_result['decision_label'] == label
    if mask.sum() > 0:
        proba_by_decision[i] = y_test_proba[mask.values].mean(axis=0)

sns.heatmap(
    proba_by_decision,
    annot=True, fmt='.2f', cmap='Blues',
    xticklabels=[l[:4] for l in LABEL_NAMES],
    yticklabels=LABEL_NAMES,
    linewidths=0.5, linecolor='white',
    ax=ax2, vmin=0, vmax=1
)
ax2.set_title('Mean P(Y=k) theo Decision')
ax2.set_xlabel('Predicted Class k')
ax2.set_ylabel('Final Decision')

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/09_percentile_proba_heatmap.png', bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIGURE_DIR}/09_percentile_proba_heatmap.png")


## 12. Demo: Đánh giá offer cho hồ sơ mới

In [ ]:
def assess_offer(
    offer_salary: float,
    pred_Q10: float, pred_Q50: float, pred_Q75: float, pred_Q90: float,
    peer_Q10: float, peer_median: float, peer_Q75: float, peer_Q90: float,
    model=None,
    cost_matrix: np.ndarray = COST_MATRIX,
    verbose: bool = True
) -> Dict:
    """
    Pipeline đánh giá offer end-to-end.
    
    Parameters
    ----------
    offer_salary : float — mức lương hiện tại của offer
    pred_Q*      : float — quantile dự đoán từ Module 1
    peer_*       : float — peer benchmark từ Module 2
    model        : sklearn model đã fit (Tầng 2)
    cost_matrix  : Cost matrix (5×5)
    verbose      : bool — in kết quả ra màn hình
    
    Returns
    -------
    dict — kết quả đầy đủ
    """
    if model is None:
        model = best_model

    # ── Bước 1: Tầng 1 — Rule Engine ─────────────────────────────────────────
    row_tmp = pd.Series({
        'offer_salary' : offer_salary,
        'pred_Q10'     : pred_Q10, 'pred_Q50': pred_Q50,
        'pred_Q75'     : pred_Q75, 'pred_Q90': pred_Q90,
        'peer_Q10'     : peer_Q10, 'peer_median': peer_median,
        'peer_Q75'     : peer_Q75, 'peer_Q90': peer_Q90,
    })
    rule_label, rule_score_val = apply_rule_engine(row_tmp)

    # ── Bước 2: Feature Engineering ──────────────────────────────────────────
    df_new   = pd.DataFrame([row_tmp])
    df_new   = build_features(df_new)
    df_new['rule_score'] = rule_score_val
    X_new    = df_new[FEATURE_COLS].values

    # ── Bước 3: Tầng 2 — Ordinal Classification ──────────────────────────────
    proba    = model.predict_proba(X_new)[0]          # (5,)

    # ── Bước 4: Tầng 3 — Cost-Sensitive Decision ─────────────────────────────
    exp_cost = proba @ cost_matrix.T                  # (5,)
    exp_util = -exp_cost
    decision_idx = int(np.argmin(exp_cost))
    decision_label = LABEL_NAMES[decision_idx]
    confidence     = float(proba.max())

    # ── Explainable Recommendation ────────────────────────────────────────────
    def _format_diff(offer, ref, ref_name):
        diff = offer - ref
        pct  = diff / ref * 100
        sign = '+' if diff >= 0 else ''
        return f"  → vs {ref_name}: {sign}${diff:,.0f} ({sign}{pct:.1f}%)"

    result = {
        'offer_salary'            : offer_salary,
        'rule_label'              : rule_label,
        'rule_score'              : rule_score_val,
        'offer_quality_label'     : decision_label,
        'offer_quality_probability': float(proba[decision_idx]),
        'decision'                : decision_label,
        'confidence_score'        : confidence,
        'expected_cost'           : float(exp_cost[decision_idx]),
        'expected_utility'        : float(exp_util[decision_idx]),
        'probabilities'           : {l: float(p) for l, p in zip(LABEL_NAMES, proba)},
        'all_expected_costs'      : {l: float(c) for l, c in zip(LABEL_NAMES, exp_cost)},
    }

    if verbose:
        print("=" * 65)
        print("  OFFER ASSESSMENT REPORT")
        print("=" * 65)
        print(f"  Offer Salary          : ${offer_salary:>12,.0f}")
        print()
        print(f"  ── Tầng 1 (Rule Engine) ──────────────────────────────")
        print(f"  pred_Q10 / Q50 / Q90  : ${pred_Q10:>10,.0f} / ${pred_Q50:>10,.0f} / ${pred_Q90:>10,.0f}")
        print(f"  peer_Q10 / med / Q90  : ${peer_Q10:>10,.0f} / ${peer_median:>10,.0f} / ${peer_Q90:>10,.0f}")
        print(f"  Rule Label            : {rule_label}  (score = {rule_score_val})")
        print()
        print(f"  ── Tầng 2 (Ordinal Classification) ───────────────────")
        for lname, prob in zip(LABEL_NAMES, proba):
            bar = '█' * int(prob * 30)
            print(f"  P({lname:<12}): {prob:.4f}  {bar}")
        print()
        print(f"  ── Tầng 3 (Cost-Sensitive Decision) ──────────────────")
        for lname, ec in zip(LABEL_NAMES, exp_cost):
            marker = ' ← DECISION' if lname == decision_label else ''
            print(f"  E[Cost | {lname:<12}]: {ec:.3f}{marker}")
        print()
        print(f"  ══ KẾT QUẢ CUỐI CÙNG ════════════════════════════════")
        print(f"  Offer Quality Label   : {decision_label}")
        print(f"  Offer Quality Prob    : {proba[decision_idx]:.4f}")
        print(f"  Confidence Score      : {confidence:.4f}  ", end='')
        if confidence >= 0.7:
            print("[CAO — Hệ thống chắc chắn]")
        elif confidence >= 0.4:
            print("[TRUNG BÌNH — Nên xem xét thêm]")
        else:
            print("[THẤP — Hồ sơ ở ranh giới]")
        print(f"  Expected Cost         : {exp_cost[decision_idx]:.3f}")
        print()
        print(f"  ── Giải thích ────────────────────────────────────────")
        print(_format_diff(offer_salary, pred_Q50, 'pred_Q50'))
        print(_format_diff(offer_salary, peer_median, 'peer_median'))
        print("=" * 65)

    return result


# ── Demo với 3 kịch bản ───────────────────────────────────────────────────────
print("\n" + "="*65)
print("KỊCH BẢN 1: Offer rõ ràng thấp")
print("="*65)
r1 = assess_offer(
    offer_salary=55_000,
    pred_Q10=72_000, pred_Q50=95_000, pred_Q75=112_000, pred_Q90=130_000,
    peer_Q10=68_000, peer_median=90_000, peer_Q75=108_000, peer_Q90=125_000,
)

print("\n" + "="*65)
print("KỊCH BẢN 2: Offer hợp lý (FAIR)")
print("="*65)
r2 = assess_offer(
    offer_salary=96_000,
    pred_Q10=72_000, pred_Q50=95_000, pred_Q75=112_000, pred_Q90=130_000,
    peer_Q10=68_000, peer_median=90_000, peer_Q75=108_000, peer_Q90=125_000,
)

print("\n" + "="*65)
print("KỊCH BẢN 3: Offer xuất sắc")
print("="*65)
r3 = assess_offer(
    offer_salary=145_000,
    pred_Q10=72_000, pred_Q50=95_000, pred_Q75=112_000, pred_Q90=130_000,
    peer_Q10=68_000, peer_median=90_000, peer_Q75=108_000, peer_Q90=125_000,
)


## 13. Lưu model và kết quả

In [ ]:
# ── Lưu models ────────────────────────────────────────────────────────────────
joblib.dump(ord_lr,  f'{OUTPUT_DIR}/offer_assessment_ordinal_lr.pkl')
joblib.dump(rf_clf,  f'{OUTPUT_DIR}/offer_assessment_rf.pkl')
joblib.dump(best_model, f'{OUTPUT_DIR}/offer_assessment_model.pkl')
print(f"Đã lưu: offer_assessment_ordinal_lr.pkl")
print(f"Đã lưu: offer_assessment_rf.pkl")
print(f"Đã lưu: offer_assessment_model.pkl  (best: {best_model_name})")

# ── Lưu predictions ────────────────────────────────────────────────────────────
df_result.to_csv(f'{OUTPUT_DIR}/offer_predictions.csv', index=False)
print(f"Đã lưu: offer_predictions.csv")

# ── Lưu probabilities riêng ───────────────────────────────────────────────────
proba_df = pd.DataFrame(y_test_proba, columns=[f'P_{l}' for l in LABEL_NAMES])
proba_df['decision']         = [LABEL_NAMES[y] for y in y_test_decision]
proba_df['confidence_score'] = conf_test
proba_df.to_csv(f'{OUTPUT_DIR}/offer_probabilities.csv', index=False)
print(f"Đã lưu: offer_probabilities.csv")

# ── Lưu Feature Importance ────────────────────────────────────────────────────
fi_df = pd.DataFrame({
    'feature'    : FEATURE_COLS,
    'importance' : rf_clf.feature_importances_
}).sort_values('importance', ascending=False)
fi_df.to_csv(f'{OUTPUT_DIR}/offer_feature_importance.csv', index=False)
print(f"Đã lưu: offer_feature_importance.csv")

# ── Lưu metrics ───────────────────────────────────────────────────────────────
metrics_dict = {
    'best_model'         : best_model_name,
    'ordinal_lr'         : metrics_ord,
    'random_forest'      : metrics_rf,
    'cost_sensitive'     : metrics_cs,
    'cost_matrix'        : COST_MATRIX.tolist(),
    'confidence_mean'    : float(conf_test.mean()),
    'confidence_median'  : float(np.median(conf_test)),
    'label_distribution' : {
        l: int((y_test_decision == i).sum()) for i, l in enumerate(LABEL_NAMES)
    }
}
with open(f'{OUTPUT_DIR}/offer_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, indent=2, ensure_ascii=False)
print(f"Đã lưu: offer_metrics.json")

print()
print("=" * 65)
print("  Tóm tắt Module 3 — Offer Assessment System")
print("=" * 65)
print(f"  Best model         : {best_model_name}")
print(f"  Accuracy (Test)    : {metrics_cs['Accuracy']:.4f}")
print(f"  Macro F1 (Test)    : {metrics_cs['Macro F1']:.4f}")
print(f"  ROC-AUC (Test)     : {metrics_cs['ROC-AUC (macro OvR)']:.4f}")
print(f"  Mean Confidence    : {conf_test.mean():.4f}")
print()
print("Files đã lưu:")
print(f"  {OUTPUT_DIR}/offer_assessment_model.pkl      — Model tốt nhất")
print(f"  {OUTPUT_DIR}/offer_assessment_ordinal_lr.pkl — Ordinal LR")
print(f"  {OUTPUT_DIR}/offer_assessment_rf.pkl         — Random Forest")
print(f"  {OUTPUT_DIR}/offer_predictions.csv           — Dự đoán tập test")
print(f"  {OUTPUT_DIR}/offer_probabilities.csv         — Xác suất 5 lớp")
print(f"  {OUTPUT_DIR}/offer_feature_importance.csv    — Feature importance")
print(f"  {OUTPUT_DIR}/offer_metrics.json              — Metrics tổng hợp")
if HAS_SHAP:
    print(f"  {OUTPUT_DIR}/offer_shap_values.csv           — SHAP values")
print("=" * 65)


## 14. Hướng dẫn tích hợp vào pipeline tổng thể

### Vị trí trong pipeline

```
Pipeline tổng thể:
┌─────────────────────────────────────────────────────────┐
│  Preprocessing → Feature Engineering                   │
│         ↓                                              │
│  [Module 1] RF/LGBM/XGB → pred_Q10/Q50/Q75/Q90        │
│         ↓                                              │
│  [Module 2] KNN → peer_Q10/median/Q75/Q90             │
│         ↓                                              │
│  [Module 3] Offer Assessment System ← Module này      │
│    • Tầng 1: Rule Engine                               │
│    • Tầng 2: Ordinal Classification                    │
│    • Tầng 3: Cost-Sensitive Decision                   │
│         ↓                                              │
│  Kết quả: offer_quality_label, confidence, decision    │
└─────────────────────────────────────────────────────────┘
```

### Cách sử dụng cho hồ sơ mới

```python
import joblib

# Load model
model = joblib.load('offer_assessment_outputs/offer_assessment_model.pkl')

# Chạy đánh giá
result = assess_offer(
    offer_salary=85_000,
    pred_Q10=70_000, pred_Q50=92_000, pred_Q75=110_000, pred_Q90=128_000,
    peer_Q10=65_000, peer_median=88_000, peer_Q75=106_000, peer_Q90=122_000,
    model=model
)

print(result['decision'])           # "FAIR"
print(result['confidence_score'])   # 0.62
print(result['expected_cost'])      # 1.45
print(result['probabilities'])      # {'UNDERPAID': 0.05, 'LOW': 0.18, 'FAIR': 0.57, ...}
```

### Điểm mạnh của kiến trúc 3 tầng

| Tầng | Vai trò | Điểm mạnh |
|---|---|---|
| Rule Engine | Gán nhãn ban đầu theo rule rõ ràng | Dễ giải thích, không cần data |
| Ordinal LR | Capture thứ tự nhãn, output xác suất | Probabilistic, calibrated |
| Cost Decision | Đưa ra quyết định tối ưu | Phản ánh đúng chi phí thực tế |

### Đề xuất cải tiến

1. **Calibration**: Dùng `CalibratedClassifierCV` để cải thiện calibration của xác suất.
2. **Threshold tuning**: Điều chỉnh cost matrix theo domain (e.g., ngành CNTT vs Finance).
3. **Ordinal LR chặt chẽ hơn**: Implement Proportional Odds Model thay vì Independent binary.
4. **Uncertainty propagation**: Propagate uncertainty từ Module 1 (pred interval width) vào feature.
